# Stage 8 — Misconception Classifier
Uses **Llama 3 (via Groq)** to map a detected error to a **misconception category**.

**Input:** incorrect step, previous (correct) step, detected operation  
**Output:** misconception category + subcategory

Prototype here, then copy `classify_misconception()` to `src/pipeline/classifier.py`.

In [1]:
%pip install groq python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import time
from groq import Groq
from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from .env
client = Groq()

# Get your free API key at: https://console.groq.com
# Add to .env: GROQ_API_KEY=your_key_here

## Misconception taxonomy
Maps to the `error_type` values in your dataset.

In [3]:
TAXONOMY = {
    "Conceptual Error": [
        "Misunderstanding of zero product rule",
        "Misunderstanding of factorization",
        "Incorrect application of algebraic identity",
        "Other conceptual misunderstanding"
    ],
    "Computational Error": [
        "Sign error",
        "Arithmetic mistake",
        "Missing root",
        "Incorrect substitution into formula"
    ],
    "Procedural Error": [
        "Incorrect factorization",
        "Incorrect zero product application",
        "Wrong sequence of steps",
        "Incomplete procedure"
    ],
    "Radical & Simplification Error": [
        "Incorrect square root simplification",
        "Discriminant calculation error",
        "Incorrect radical arithmetic"
    ],
    "Common Factor Error": [
        "Incorrect extraction of common factor",
        "Missed common factor"
    ]
}

# Map dataset error_type strings → taxonomy categories
DATASET_LABEL_MAP = {
    "conceptual_sign_error":        "Computational Error",
    "computational_error":          "Computational Error",
    "procedural_error":             "Procedural Error",
    "radical_simplification_error": "Radical & Simplification Error",
    "common_factor_error":          "Common Factor Error"
}

taxonomy_str = "\n".join(
    f"- {cat}: {', '.join(subs)}"
    for cat, subs in TAXONOMY.items()
)
print(taxonomy_str)

- Conceptual Error: Misunderstanding of zero product rule, Misunderstanding of factorization, Incorrect application of algebraic identity, Other conceptual misunderstanding
- Computational Error: Sign error, Arithmetic mistake, Missing root, Incorrect substitution into formula
- Procedural Error: Incorrect factorization, Incorrect zero product application, Wrong sequence of steps, Incomplete procedure
- Radical & Simplification Error: Incorrect square root simplification, Discriminant calculation error, Incorrect radical arithmetic
- Common Factor Error: Incorrect extraction of common factor, Missed common factor


## Build the prompt

In [4]:
SYSTEM_PROMPT = f"""Classify algebra errors using this taxonomy: {taxonomy_str}

Examples:
- "x^2-5x+6=0" → "(x-2)(x+3)=0" during Factorization = {{"category": "Procedural Error", "subcategory": "Incorrect factorization", "confidence": 0.95}}
- "(x-2)(x-3)=0" → "x=2" during Zero Product Rule = {{"category": "Computational Error", "subcategory": "Missing root", "confidence": 0.97}}
- "x^2-5x+6=0" → "x=(5±√1)/2" during Quadratic Formula = {{"category": "Radical & Simplification Error", "subcategory": "Discriminant calculation error", "confidence": 0.92}}

Now classify:
Previous step: {{step_prev}}
Wrong step: {{step_wrong}}
Operation: {{operation}}

Respond ONLY with JSON: {{"category": "...", "subcategory": "...", "confidence": <0.0-1.0>}}"""


def classify_misconception(
    step_prev: str,
    step_wrong: str,
    operation: str,
    retries: int = 3
) -> dict:
    """
    Classify the misconception for an incorrect step using Llama 3 via Groq.
    Returns: {"category": str, "subcategory": str, "confidence": float}
    """
    user_msg = f"""Correct previous step: {step_prev}
Incorrect step:        {step_wrong}
Operation attempted:   {operation}

What type of error is this?"""

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg}
                ],
                temperature=0
            )
            raw = response.choices[0].message.content.strip()
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"): raw = raw[4:]
                raw = raw.strip()
            return json.loads(raw)

        except Exception as e:
            if attempt < retries - 1:
                wait = 2 ** attempt
                print(f"  Attempt {attempt+1} failed: {e}. Retrying in {wait}s...")
                time.sleep(wait)
            else:
                return {"category": "Conceptual Error",
                        "subcategory": "Other conceptual misunderstanding",
                        "confidence": 0.0}

## Test cases

In [52]:
# Test 1: Wrong sign in factorization
result = classify_misconception(
    step_prev="x^2 - 5x + 6 = 0",
    step_wrong="(x - 2)(x + 3) = 0",
    operation="Factorization"
)
print("Test 1 (expect Procedural/Incorrect factorization):")
print(json.dumps(result, indent=2))

Test 1 (expect Procedural/Incorrect factorization):
{
  "category": "Procedural Error",
  "subcategory": "Incorrect factorization",
  "confidence": 0.95
}


In [40]:
# Test 2: Missing root
result = classify_misconception(
    step_prev="(x - 2)(x - 3) = 0",
    step_wrong="x = 2",
    operation="Apply Zero Product Rule"
)
print("Test 2 (expect Computational/Missing root):")
print(json.dumps(result, indent=2))

Test 2 (expect Computational/Missing root):
{
  "category": "Computational Error",
  "subcategory": "Missing root",
  "confidence": 0.97
}


In [41]:
# Test 3: Radical simplification error (from your dataset: √12 = 12)
result = classify_misconception(
    step_prev="x = (5 ± √(25 - 24)) / 2",
    step_wrong="√12 = 12",
    operation="Simplify Radical"
)
print("Test 3 (expect Radical/Incorrect square root simplification):")
print(json.dumps(result, indent=2))

Test 3 (expect Radical/Incorrect square root simplification):
{
  "category": "Radical & Simplification Error",
  "subcategory": "Incorrect square root simplification",
  "confidence": 0.99
}


In [42]:
# Test 4: Discriminant error (from your dataset: 0 - -144 = 145)
result = classify_misconception(
    step_prev="x = (-(0) ± √((0)^2 - 4(3)(-12)))/(2*3)",
    step_wrong="0 - -144 = 145",
    operation="Calculate Discriminant"
)
print("Test 4 (expect Radical/Discriminant calculation error):")
print(json.dumps(result, indent=2))

Test 4 (expect Radical/Discriminant calculation error):
{
  "category": "Computational Error",
  "subcategory": "Arithmetic mistake",
  "confidence": 0.98
}


## Batch eval against dataset ground truth

In [5]:
import pandas as pd
import re
from sympy import symbols, expand, simplify
from sympy.parsing.sympy_parser import parse_expr, standard_transformations

DATASET_PATH = r"C:\mariam\uni\bachelor\algebra-error-detector\notebooks\quadratic_dataset.json"

with open(DATASET_PATH) as f:
    dataset = json.load(f)

x = symbols("x")

# ── Sympy helpers (same as error_detector.ipynb) ────────────────────────────
def to_sympy_notation(expr):
    expr = re.sub(r"=\s*0", "", expr).strip()
    expr = expr.replace("^", "**")
    expr = re.sub(r"\)\s*\(", ")*(" , expr)
    expr = re.sub(r"(\d)\s*\(", r"\1*(", expr)
    expr = re.sub(r"(\d)\s*([a-zA-Z])", r"\1*\2", expr)
    return expr.strip()

def validate_arithmetic(expr):
    try:
        m = re.match(r"^([\d\s\+\-\*\/\.]+)=([\d\s\.]+)$", expr.strip())
        if m: return abs(eval(m.group(1)) - float(m.group(2))) < 0.01
        m = re.match(r"^√(\d+)\s*=\s*([\d\.]+)$", expr.strip())
        if m:
            import math
            return abs(math.sqrt(float(m.group(1))) - float(m.group(2))) < 0.01
        return None
    except: return None

def validate_factorization(equation, factored):
    try:
        orig = parse_expr(to_sympy_notation(equation), transformations=standard_transformations, local_dict={"x": x})
        fact = parse_expr(to_sympy_notation(factored), transformations=standard_transformations, local_dict={"x": x})
        return simplify(expand(fact) - orig) == 0
    except: return None

def find_first_error(entry):
    steps, equation = entry["steps"], entry["equation"]
    for i in range(1, len(steps)):
        prev, curr = steps[i-1], steps[i]
        if validate_arithmetic(curr) is False:
            return {"step_prev": prev, "step_wrong": curr, "operation": "Calculate Discriminant"}
        if "(" in curr and ")" in curr and "OR" not in curr and "±" not in curr:
            if validate_factorization(equation, curr) is False:
                return {"step_prev": prev, "step_wrong": curr, "operation": "Factorization"}
    return None

# Use sympy to find error entries — no labels needed
error_entries = []
for e in dataset:
    err = find_first_error(e)
    if err:
        error_entries.append({**e, **err})

print(f"Total entries:              {len(dataset)}")
print(f"Entries with detected error: {len(error_entries)}")


Total entries:              2100
Entries with detected error: 1558


In [6]:
rows = []
for i, entry in enumerate(error_entries[:30]):
    result = classify_misconception(
        step_prev=entry["step_prev"],
        step_wrong=entry["step_wrong"],
        operation=entry["operation"]
    )
    rows.append({
        "equation":      entry["equation"],
        "step_wrong":    entry["step_wrong"],
        "pred_category": result.get("category"),
        "subcategory":   result.get("subcategory"),
        "confidence":    result.get("confidence"),
    })
    if (i + 1) % 10 == 0:
        time.sleep(3)

df = pd.DataFrame(rows)
df


,equation,step_wrong,pred_category,subcategory,confidence
0,1x^2 + -6x + 0 = 0,36 - 0 = 40,Radical & Simplification Error,Discriminant calculation error,0.99
1,1x^2 + -6x + 0 = 0,√12 = 12,Radical & Simplification Error,Incorrect square root simplification,0.99
2,1x^2 + -6x + 0 = 0,(x + 1)(x + 4) = 0,Procedural Error,Incorrect factorization,0.96
3,1x^2 + -5x + -50 = 0,25 - -200 = 229,Radical & Simplification Error,Discriminant calculation error,0.98
4,1x^2 + -5x + -50 = 0,√12 = 12,Radical & Simplification Error,Incorrect square root simplification,0.99
5,1x^2 + -5x + -50 = 0,(x + 1)(x + 4) = 0,Procedural Error,Incorrect factorization,0.98
6,3x^2 + 36x + 96 = 0,(x - -8)(x - -4) = 0,Procedural Error,Incorrect factorization,0.96
7,3x^2 + 36x + 96 = 0,(x - -8)(x + 4) = 0,Procedural Error,Incorrect factorization,0.96
8,3x^2 + 36x + 96 = 0,1296 - 1152 = 149,Radical & Simplification Error,Discriminant calculation error,0.96
9,3x^2 + 36x + 96 = 0,(x - -8)(x - -4) = 0,Procedural Error,Incorrect factorization,0.95


In [7]:
# Distribution of predicted categories

print(f"\nAverage confidence: {df['confidence'].mean():.2f}")
print(f"Low confidence (<0.8): {len(df[df['confidence'] < 0.8])} rows")

# Subcategory breakdown
print("\nSubcategory distribution:")
print(df["subcategory"].value_counts())


Average confidence: 0.97
Low confidence (<0.8): 0 rows

Subcategory distribution:
subcategory
Incorrect factorization                 17
Incorrect square root simplification     6
Discriminant calculation error           5
Arithmetic mistake                       2
Name: count, dtype: int64


## ✅ Once prompt is good → copy `classify_misconception()` to `src/pipeline/classifier.py`